# CIR Monte Carlo Quiz Solver
This notebook cell implements the 4 required steps using a lightweight OOP structure.
Edit the parameters at the top of the code cell before running during the quiz.

In [ ]:
import random
from dataclasses import dataclass
import numpy as np

# =============================
# Quiz parameters 
# =============================
kappa = 0.1
theta = 0.05
sigma = 0.08
r0 = 0.04
strike = 0.04
seed_value = 123


@dataclass
class CIRConfig:
    kappa: float
    theta: float
    sigma: float
    r0: float
    strike: float
    seed_value: int
    n_paths: int = 100000
    n_days: int = 365
    dt: float = 1.0 / 365.0
    reset_day: int = 275


class CIRMonteCarlo:
    def __init__(self, cfg: CIRConfig):
        self.cfg = cfg

    def simulate_paths(self) -> np.ndarray:
        """Step 1: Euler simulation with max(r[t-1], 0) only inside sqrt."""
        random.seed(self.cfg.seed_value)
        np.random.seed(self.cfg.seed_value)

        r = np.zeros((self.cfg.n_paths, self.cfg.n_days + 1), dtype=float)
        r[:, 0] = self.cfg.r0

        sqrt_dt = np.sqrt(self.cfg.dt)
        for t in range(1, self.cfg.n_days + 1):
            z = np.random.normal(0.0, 1.0, self.cfg.n_paths)
            r_prev = r[:, t - 1]
            drift = self.cfg.kappa * (self.cfg.theta - r_prev) * self.cfg.dt
            diffusion = self.cfg.sigma * np.sqrt(np.maximum(r_prev, 0.0)) * sqrt_dt * z
            r[:, t] = r_prev + drift + diffusion
        return r

    def discount_factors(self, paths: np.ndarray) -> np.ndarray:
        """Step 3: DF = exp(-sum(r[0:365] * dt)) per path."""
        integral = np.sum(paths[:, 0:self.cfg.n_days] * self.cfg.dt, axis=1)
        return np.exp(-integral)


class CapletPricer:
    def __init__(self, cfg: CIRConfig):
        self.cfg = cfg
        # tau is the fraction of the year between reset (day 275) and settlement (day 365)
        self.tau = (self.cfg.n_days - self.cfg.reset_day) / 365.0 

    def forward_looking_premium_bps(self, paths: np.ndarray, discount_factors: np.ndarray) -> float:
        """Step 3: payoff = max(r_reset - strike, 0), discounted and averaged."""
        r_reset = paths[:, self.cfg.reset_day]
        payoff = self.tau * np.maximum(r_reset - self.cfg.strike, 0.0)
        premium = np.mean(discount_factors * payoff)
        return 10000.0 * premium

    def backward_looking_premium_bps(self, paths: np.ndarray, discount_factors: np.ndarray) -> float:
        """Step 4: payoff = max(mean(r[275:366]) - strike, 0), discounted and averaged."""
        r_bar = np.mean(paths[:, self.cfg.reset_day:self.cfg.n_days + 1], axis=1)
        payoff = self.tau * np.maximum(r_bar - self.cfg.strike, 0.0)
        premium = np.mean(discount_factors * payoff)
        return 10000.0 * premium


def print_descriptive_statistics(final_rates: np.ndarray) -> None:
    """Step 2: mean, std, 25%, 50%, 75% for rates at day 365."""
    stats = {
        "mean": np.mean(final_rates),
        "std": np.std(final_rates, ddof=1),
        "q25": np.quantile(final_rates, 0.25),
        "median": np.quantile(final_rates, 0.50),
        "q75": np.quantile(final_rates, 0.75),
    }
    print("Step 2 - Descriptive Statistics (Day 365 rates)")
    for k, v in stats.items():
        print(f"{k:>6}: {v:.6f}")


cfg = CIRConfig(
    kappa=kappa,
    theta=theta,
    sigma=sigma,
    r0=r0,
    strike=strike,
    seed_value=seed_value,
 )

simulator = CIRMonteCarlo(cfg)
paths = simulator.simulate_paths()

# Step 2
final_rates = paths[:, cfg.n_days]
print_descriptive_statistics(final_rates)

# Step 3 and Step 4
dfs = simulator.discount_factors(paths)
pricer = CapletPricer(cfg)

fw_premium_bps = pricer.forward_looking_premium_bps(paths, dfs)
bw_premium_bps = pricer.backward_looking_premium_bps(paths, dfs)

print("\nStep 3 - Forward-looking caplet premium (bps)")
print(f"{fw_premium_bps:.6f}")

print("\nStep 4 - Backward-looking caplet premium (bps)")
print(f"{bw_premium_bps:.6f}")

Step 2 - Descriptive Statistics (Day 365 rates)
  mean: 0.041006
   std: 0.015274
   q25: 0.030044
median: 0.039558
   q75: 0.050311

Step 3 - Forward-looking caplet premium (bps)
13.337653

Step 4 - Backward-looking caplet premium (bps)
13.988610


In [ ]:
from dataclasses import dataclass
import numpy as np

# ===============================================
# Quiz parameters for Q4/Q5
# ===============================================
kappa = 0.1
theta = 0.05
sigma = 0.08
r0 = 0.04
strike = 0.95


@dataclass
class CIRFDConfig:
    kappa: float
    theta: float
    sigma: float
    r0: float
    strike: float
    dr: float = 0.001
    imax: int = 100
    dt: float = 1.0 / 2000.0
    N_time: int = 2000
    T_put: float = 1.0
    T_bond: float = 3.0


class CIREFDPutPricer:
    def __init__(self, cfg: CIRFDConfig):
        self.cfg = cfg
        self.r_grid = np.arange(cfg.imax + 1) * cfg.dr

    def P_cir(self, t: float, T: float, r):
        tau = T - t
        gamma = np.sqrt(self.cfg.kappa**2 + 2.0 * self.cfg.sigma**2)
        exp_gt = np.exp(gamma * tau)
        den = (self.cfg.kappa + gamma) * (exp_gt - 1.0) + 2.0 * gamma
        a_cir = ((2.0 * gamma * np.exp((self.cfg.kappa + gamma) * tau / 2.0)) / den) ** (
            2.0 * self.cfg.kappa * self.cfg.theta / self.cfg.sigma**2
        )
        b_cir = (2.0 * (exp_gt - 1.0)) / den
        return a_cir * np.exp(-b_cir * r)

    def price_european_american_puts_bps(self):
        # Step 1: grid parameters and CIR bond helper already set in config and P_cir
        v1 = self.cfg.dt / (self.cfg.dr ** 2)
        v2 = self.cfg.dt / self.cfg.dr
        mu_i = self.cfg.kappa * (self.cfg.theta - self.r_grid)
        sig_i_sq = (self.cfg.sigma ** 2) * self.r_grid
        A = 0.5 * (sig_i_sq * v1 - mu_i * v2)
        B = 1.0 - sig_i_sq * v1 - self.r_grid * self.cfg.dt
        C = 0.5 * (mu_i * v2 + sig_i_sq * v1)

        # Step 2: initialize terminal payoff at t = T_put
        payoff_T_put = np.maximum(
            self.cfg.strike - self.P_cir(self.cfg.T_put, self.cfg.T_bond, self.r_grid),
            0.0,
        )
        V_eur = payoff_T_put.copy()
        V_am = payoff_T_put.copy()

        # Step 3: backward induction
        for step in range(self.cfg.N_time - 1, -1, -1):
            t = step * self.cfg.dt

            V_eur_new = np.zeros_like(V_eur)
            V_am_new = np.zeros_like(V_am)

            V_eur_new[1:self.cfg.imax] = (
                A[1:self.cfg.imax] * V_eur[0:self.cfg.imax - 1]
                + B[1:self.cfg.imax] * V_eur[1:self.cfg.imax]
                + C[1:self.cfg.imax] * V_eur[2:self.cfg.imax + 1]
            )
            V_am_new[1:self.cfg.imax] = (
                A[1:self.cfg.imax] * V_am[0:self.cfg.imax - 1]
                + B[1:self.cfg.imax] * V_am[1:self.cfg.imax]
                + C[1:self.cfg.imax] * V_am[2:self.cfg.imax + 1]
            )

            bc_low = max(
                0.0,
                self.cfg.strike * self.P_cir(t, self.cfg.T_put, 0.0)
                - self.P_cir(t, self.cfg.T_bond, 0.0),
            )
            bc_high = max(
                0.0,
                self.cfg.strike * self.P_cir(t, self.cfg.T_put, self.r_grid[self.cfg.imax])
                - self.P_cir(t, self.cfg.T_bond, self.r_grid[self.cfg.imax]),
            )
            V_eur_new[0] = bc_low
            V_eur_new[self.cfg.imax] = bc_high
            V_am_new[0] = bc_low
            V_am_new[self.cfg.imax] = bc_high

            exercise_value = np.maximum(
                0.0,
                self.cfg.strike - self.P_cir(t, self.cfg.T_bond, self.r_grid),
            )
            V_am_new = np.maximum(V_am_new, exercise_value)

            V_eur = V_eur_new
            V_am = V_am_new

        # Step 4: interpolate at r0 and convert to bps
        eur_premium = np.interp(self.cfg.r0, self.r_grid, V_eur)
        am_premium = np.interp(self.cfg.r0, self.r_grid, V_am)

        eur_bps = 10000.0 * eur_premium
        am_bps = 10000.0 * am_premium
        return eur_bps, am_bps


fd_cfg = CIRFDConfig(
    kappa=kappa,
    theta=theta,
    sigma=sigma,
    r0=r0,
    strike=strike,
 )

efd_pricer = CIREFDPutPricer(fd_cfg)
eur_put_bps, am_put_bps = efd_pricer.price_european_american_puts_bps()

print("Q4 - European put premium on 3Y ZCB (1Y expiry), EFD under CIR [bps]")
print(f"{eur_put_bps:.6f}")

print("\nQ5 - American put premium on 3Y ZCB (1Y expiry), EFD under CIR [bps]")
print(f"{am_put_bps:.6f}")

Q4 - European put premium on 3Y ZCB (1Y expiry), EFD under CIR [bps]
291.543530

Q5 - American put premium on 3Y ZCB (1Y expiry), EFD under CIR [bps]
658.676012
